In [ ]:
import warnings
warnings.simplefilter(action='ignore')
import pycisTopic
pycisTopic.__version__

In [ ]:
projDir = '/mnt/My_disk/Li_linghang/liumeng_HDCAC1/dataset/output/proj_d16new_d19wt/pycistopic_clusters2/'
outDir = projDir + 'output/'
import os
if not os.path.exists(outDir):
    os.makedirs(outDir)

In [ ]:
tmpDir = projDir + 'output/tmp/'
if not os.path.exists(tmpDir):
    os.makedirs(tmpDir)

In [ ]:
from pycisTopic.cistopic_class import *
count_matrix=pd.read_csv(projDir+'pkm.tsv', sep='\t')

In [ ]:
path_to_blacklist='/mnt/My_disk/Li_linghang/Wangyang_cervix_D16_D19/scenicplus/mm10-blacklist.v2.bed'####这个写死不要动
cistopic_obj = create_cistopic_object(fragment_matrix=count_matrix, path_to_blacklist=path_to_blacklist)
# Adding cell information
cell_data =  pd.read_csv(projDir+'celldata.tsv', sep='\t')
cistopic_obj.add_cell_data(cell_data)

In [ ]:
import os
import pycisTopic
import pandas as pd
import polars as pl
import pickle
from pycisTopic.pseudobulk_peak_calling import export_pseudobulk, peak_calling
from pycisTopic.iterative_peak_calling import get_consensus_peaks
from pycisTopic.cistopic_class import create_cistopic_object_from_fragments
from pycisTopic.lda_models import run_cgs_models_mallet, evaluate_models
from pycisTopic.clust_vis import find_clusters, run_umap, run_tsne, plot_metadata
from pycisTopic.topic_binarization import binarize_topics
from pycisTopic.topic_qc import compute_topic_metrics
from pycisTopic.diff_features import impute_accessibility, find_diff_features
from pycisTopic.gene_activity import get_gene_activity

In [ ]:
import os
os.environ['MALLET_HEAP_SIZE'] = '256G' 

In [ ]:
path_to_mallet_binary='/home/lilinghang/Mallet-202108/bin/mallet'
# Run models
models=run_cgs_models_mallet(
  mallet_path = path_to_mallet_binary,
  cistopic_obj = cistopic_obj,
  n_topics=[2,5,10,15,20,25,30],
  n_cpu=20,
  n_iter=500,
  random_state=1,
  alpha=50,
  alpha_by_topic=True,
  eta=0.1,
  eta_by_topic=False,
  tmp_path= tmpDir, #Use SCRATCH if many models or big data set
  save_path = None)

In [ ]:
import pickle 
with open(outDir+'Mallet_models_500.pkl', 'wb') as f:
  pickle.dump(models, f)

In [ ]:
os.mkdir(outDir+'/models')

In [ ]:
model=evaluate_models(models,
                     select_model=None, 
                     return_model=True, 
                     metrics=['Arun_2010','Cao_Juan_2009', 'Minmo_2011', 'loglikelihood'],
                     plot_metrics=False,
                     save= outDir + 'models/model_selection.pdf')

In [ ]:
cistopic_obj.add_LDA_model(model)

In [ ]:
with open(outDir + 'D19_cisTopicObject.pkl', 'wb') as f:
  pickle.dump(cistopic_obj, f)

In [ ]:
import pickle
infile = open(outDir + 'D19_cisTopicObject.pkl', 'rb')
cistopic_obj = pickle.load(infile)
infile.close()

In [ ]:
from pycisTopic.clust_vis import *
find_clusters(cistopic_obj,
                 target  = 'cell',
                 k = 10,
                 res = [0.1],
                 prefix = 'pycisTopic_', 
                 scale = True,
                 split_pattern = '-')

In [ ]:
run_umap(cistopic_obj,
         target='cell', 
         scale=True, 
         reduction_name='UMAP')

In [ ]:
os.mkdir(outDir+'/visualization')

In [ ]:
plot_topic(cistopic_obj,
            reduction_name = 'UMAP',
            target = 'cell',
            num_columns=5,
            save= outDir + 'visualization/dimensionality_reduction_topic_contr.pdf')

In [ ]:
with open(outDir + 'D19_cisTopicObject.pkl', 'wb') as f:
  pickle.dump(cistopic_obj, f)

In [ ]:
from pycisTopic.topic_binarization import *
region_bin_topics_top_3k = binarize_topics(
    cistopic_obj, method='ntop', ntop = 3_000,
    plot=True, num_columns=5
)

In [ ]:
region_bin_topics_otsu = binarize_topics(
    cistopic_obj, method='otsu',
    plot=True, num_columns=5
)

In [ ]:
binarized_cell_topic = binarize_topics(
    cistopic_obj,
    target='cell',
    method='li',
    plot=True,
    num_columns=5, nbins=100)

In [ ]:
from pycisTopic.topic_qc import compute_topic_metrics, plot_topic_qc, topic_annotation
import matplotlib.pyplot as plt
from pycisTopic.utils import fig2img
topic_qc_metrics = compute_topic_metrics(cistopic_obj)

In [ ]:
fig_dict={}
fig_dict['CoherenceVSAssignments']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Log10_Assignments', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['AssignmentsVSCells_in_bin']=plot_topic_qc(topic_qc_metrics, var_x='Log10_Assignments', var_y='Cells_in_binarized_topic', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['CoherenceVSCells_in_bin']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Cells_in_binarized_topic', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['CoherenceVSRegions_in_bin']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Regions_in_binarized_topic', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['CoherenceVSMarginal_dist']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Marginal_topic_dist', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['CoherenceVSGini_index']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Gini_index', var_color='Gini_index', plot=False, return_fig=True)

In [ ]:
fig=plt.figure(figsize=(40, 43))
i = 1
for fig_ in fig_dict.keys():
    plt.subplot(2, 3, i)
    img = fig2img(fig_dict[fig_]) #To convert figures to png to plot together, see .utils.py. This converts the figure to png.
    plt.imshow(img)
    plt.axis('off')
    i += 1
plt.subplots_adjust(wspace=0, hspace=-0.70)
plt.show()



In [ ]:
topic_annot = topic_annotation(
    cistopic_obj,
    annot_var='Sample_Clusters2',
    binarized_cell_topic=binarized_cell_topic,
    general_topic_thr = 0.2
)

In [ ]:
topic_annot

In [ ]:
from pycisTopic.diff_features import (
    impute_accessibility,
    normalize_scores,
    find_highly_variable_features,
    find_diff_features
)
import numpy as np

In [ ]:
imputed_acc_obj = impute_accessibility(
    cistopic_obj,
    selected_cells=None,
    selected_regions=None,
    scale_factor=10**6
)

In [ ]:
normalized_imputed_acc_obj = normalize_scores(imputed_acc_obj, scale_factor=10**4)

In [ ]:
variable_regions = find_highly_variable_features(
    normalized_imputed_acc_obj,
    min_disp = 0.05,
    min_mean = 0.0125,
    max_mean = 3,
    max_disp = np.inf,
    n_bins=20,
    n_top_features=None,
    plot=True
)

In [ ]:
tmpDir = "/tmp/ray_spill/"
markers_dict= find_diff_features(
    cistopic_obj,
    imputed_acc_obj,
    variable='Sample_Clusters2',
    var_features=variable_regions,
    contrasts=None,
    adjpval_thr=0.05,
    log2fc_thr=np.log2(1.5),
    n_cpu=5,
    _temp_dir=tmpDir + 'ray_spill',
    split_pattern = '-'
)

In [ ]:
os.makedirs(os.path.join(outDir, "region_sets"), exist_ok = True)
os.makedirs(os.path.join(outDir, "region_sets", "Topics_otsu"), exist_ok = True)
os.makedirs(os.path.join(outDir, "region_sets", "Topics_top_3k"), exist_ok = True)
os.makedirs(os.path.join(outDir, "region_sets", "DARs_cell_type"), exist_ok = True)

In [ ]:
for topic in region_bin_topics_otsu:
    region_names_to_coordinates(
        region_bin_topics_otsu[topic].index
    ).sort_values(
        ["Chromosome", "Start", "End"]
    ).to_csv(
        os.path.join(outDir, "region_sets", "Topics_otsu", f"{topic}.bed"),
        sep = "\t",
        header = False, index = False
    )

In [ ]:
for topic in region_bin_topics_top_3k:
    region_names_to_coordinates(
        region_bin_topics_top_3k[topic].index
    ).sort_values(
        ["Chromosome", "Start", "End"]
    ).to_csv(
        os.path.join(outDir, "region_sets", "Topics_top_3k", f"{topic}.bed"),
        sep = "\t",
        header = False, index = False
    )

In [ ]:
for cell_type in markers_dict:
    region_names_to_coordinates(
        markers_dict[cell_type].index
    ).sort_values(
        ["Chromosome", "Start", "End"]
    ).to_csv(
        os.path.join(outDir, "region_sets", "DARs_cell_type", f"{cell_type}.bed"),
        sep = "\t",
        header = False, index = False
    )

In [ ]:
fragments_dict = {
    "D16WT":'/mnt/My_disk/Li_linghang/liumeng_HDCAC1/dataset/D16_newdata/fragments_corrected_dedup_count.tsv.gz',
    "D19WT1": "/mnt/My_disk/Li_linghang/liumeng_HDCAC1/dataset/D19WT/fragments_corrected_dedup_count.tsv.gz"
}

In [ ]:
chromsizes = pd.read_table(
    "http://hgdownload.cse.ucsc.edu/goldenPath/mm10/bigZips/mm10.chrom.sizes",
    header=None, 
    names=["Chromosome", "End"]
)
chromsizes.insert(1, "Start", 0)

In [ ]:
cell_data = pd.read_table("/mnt/My_disk/Li_linghang/liumeng_HDCAC1/dataset/output/proj_d16new_d19wt/pycistopic_clusters2/celldata.tsv", index_col=0)
cell_data.index = cell_data.index.str.split('#').str[-1] ###把barcode前面的样本名去掉

In [ ]:
os.chdir('/mnt/My_disk/Li_linghang/liumeng_HDCAC1/dataset/output/proj_d16new_d19wt/pycistopic_clusters2/')
out_dir = "outs_pycisTopic_consensus_regions"
os.makedirs(out_dir, exist_ok=True)

In [ ]:
os.makedirs(os.path.join(out_dir, "consensus_peak_calling/pseudobulk_bed_files"), exist_ok=True)
os.makedirs(os.path.join(out_dir, "consensus_peak_calling/pseudobulk_bw_files"), exist_ok=True)

In [ ]:
bw_paths, bed_paths = export_pseudobulk(
    input_data=cell_data,
    variable="Clusters2",  
    sample_id_col="Sample",  
    chromsizes=chromsizes,
    bed_path=os.path.join(out_dir, "consensus_peak_calling/pseudobulk_bed_files"),
    bigwig_path=os.path.join(out_dir, "consensus_peak_calling/pseudobulk_bw_files"),
    path_to_fragments=fragments_dict,
    n_cpu=10,
    normalize_bigwig=True,
    temp_dir="/mnt/My_disk/Li_linghang/Wangyang_cervix_D16_D19/scenicplus/outs_pycisTopic_consensus_regions/tmp",
    split_pattern="-"  
)

In [ ]:
with open(os.path.join(out_dir, "consensus_peak_calling/bw_paths.tsv"), "wt") as f:
    for v in bw_paths:
        _ = f.write(f"{v}\t{bw_paths[v]}\n")

with open(os.path.join(out_dir, "consensus_peak_calling/bed_paths.tsv"), "wt") as f:
    for v in bed_paths:
        _ = f.write(f"{v}\t{bed_paths[v]}\n")##路径写出方便读取

In [ ]:
bw_paths = {}
with open(os.path.join(out_dir, "consensus_peak_calling/bw_paths.tsv")) as f:
    for line in f:
        v, p = line.strip().split("\t")
        bw_paths.update({v: p})
bed_paths = {}
with open(os.path.join(out_dir, "consensus_peak_calling/bed_paths.tsv")) as f:
    for line in f:
        v, p = line.strip().split("\t")
        bed_paths.update({v: p})

In [ ]:
os.makedirs(os.path.join(out_dir, "consensus_peak_calling/MACS"), exist_ok = True)
macs_path ='/home/lilinghang/anaconda3/envs/macs2/bin/macs2'
narrow_peak_dict = peak_calling(
    macs_path = macs_path,
    bed_paths = bed_paths,
    outdir = os.path.join(os.path.join(out_dir, "consensus_peak_calling/MACS")),
    genome_size = 'mm',
    n_cpu = 10,
    input_format = 'BEDPE',
    shift = 73,
    ext_size = 146,
    keep_dup = 'all',
    q_value = 0.05,
    _temp_dir = '/mnt/My_disk/Li_linghang/peakcallingf_tem'
)

In [ ]:
consensus_peaks = get_consensus_peaks(
    narrow_peaks_dict=narrow_peak_dict,
    peak_half_width=250,
    chromsizes=chromsizes,
    path_to_blacklist="/mnt/My_disk/Li_linghang/Wangyang_cervix_D16_D19/scenicplus/mm10-blacklist.v2.bed"
)

# 保存consensus peaks
consensus_peaks.to_bed(
    path=os.path.join(out_dir, "consensus_peak_calling/consensus_regions.bed"),
    keep=True,
    compression='infer',
    chain=False
)